# 3장 예제 — 3.7 전처리 파이프라인 정리

- **교재 출처**: manuscript/30_ch3_전처리.md
- **실행 방법**: example 폴더에서  python "3장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 04_3-4_중복·형변환·단위_정리.py, 05_3-5_스케일링.py, 06_3-6_시계열_데이터_다루기_기초.py
- **실행 상태**: 단독 실행 확인됨(선행 절 준비 코드 포함).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

In [ ]:
# --- 실행 준비 (교재에서는 앞 절에서 이미 실행한 코드) --------------------
import pandas as pd
import numpy as np

### 📌 [선행 절 준비 코드 — 단독 실행용] 아래는 3.1절에서 만든 객체를 이 파일만으로 재현한 것이다. 교재 본문에서는 앞 절에서 이미 만들어져 있으므로 다시 실행할 필요가 없다.

> 앞 절에서 만든 변수/객체를 재현하는 코드입니다.

In [ ]:
# 3.2~3.4절에서 df 에 가한 수정(믹싱_온도·건조로_1구간_온도 대체, 불량_여부 int8 변환)은
# 이 절이 쓰는 '프레스_Gap'·'불량_여부' 집계 결과를 바꾸지 않으므로 원본 적재만 재현한다.
df = pd.read_csv('data/battery_process_data.csv')
# ==================== [선행 절 준비 코드 끝] ==========================

### 재사용 가능한 clean_data() 함수

In [ ]:
def clean_data(path):
    """배터리 공정 데이터 정제 파이프라인"""
    df = pd.read_csv(path)
    # 1) 에러 코드 -> NaN
    df['건조로_1구간_온도'] = df['건조로_1구간_온도'].replace(9999, np.nan)
    # 2) 물리 범위 위반 -> NaN
    rules = {'믹싱_온도': (10, 50), '건조로_1구간_온도': (80, 160),
             '프레스_압력': (20, 80)}
    for col, (lo, hi) in rules.items():
        df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
    # 3) 결측치 -> 중앙값 대체
    num_cols = df.select_dtypes('number').columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    # 4) 중복 제거
    df = df.drop_duplicates(subset='Lot_ID')
    return df

raw = pd.read_csv('data/battery_process_data.csv')
clean = clean_data('data/battery_process_data.csv')
print("정제 전:", raw.shape, "/ 정제 후:", clean.shape)

# 교재 실행 결과 ------------------------------------------------------
#   정제 전: (1000, 10) / 정제 후: (1000, 10)

### 처리 전후 비교 — describe로 검증하기

In [ ]:
cols = ['믹싱_온도', '건조로_1구간_온도', '프레스_압력']
행 = ['count', 'mean', 'std', 'min', 'max']

print("[정제 전]")
print(raw[cols].describe().loc[행].round(2))
print()                                    # 빈 줄 하나 — 두 표를 눈으로 분리
print("[정제 후]")
print(clean[cols].describe().loc[행].round(2))

# 교재 실행 결과 ------------------------------------------------------
#   [정제 전]
#           믹싱_온도  건조로_1구간_온도   프레스_압력
#   count  995.00     1000.00  1000.00
#   mean    24.88      159.44    39.91
#   std      1.52      697.87     3.98
#   min     20.40       99.00    28.00
#   max     29.40     9999.00    52.00
#   [정제 후]
#            믹싱_온도  건조로_1구간_온도   프레스_압력
#   count  1000.00     1000.00  1000.00
#   mean     24.88      109.99    39.91
#   std       1.52        3.90     3.98
#   min      20.40       99.00    28.00
#   max      29.40      122.40    52.00

### 중간 점검 문항 (필수, 10분)

In [ ]:
raw = pd.read_csv('data/battery_process_data.csv')
print("평균 대체 후 평균:", round(raw['믹싱_온도'].fillna(raw['믹싱_온도'].mean()).mean(), 4))
print("중앙값 대체 후 평균:", round(raw['믹싱_온도'].fillna(raw['믹싱_온도'].median()).mean(), 4))

# 교재 실행 결과 ------------------------------------------------------
#   평균 대체 후 평균: 24.8791
#   중앙값 대체 후 평균: 24.8792

In [ ]:
g = df['프레스_Gap']
q1, q3 = g.quantile(0.25), g.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out = df[(g < lower) | (g > upper)]
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"하한={lower:.2f}, 상한={upper:.2f}")
print("이상치 후보:", len(out), "건 / 그중 불량:", int(out['불량_여부'].sum()))

# 교재 실행 결과 ------------------------------------------------------
#   Q1=91.20, Q3=99.30, IQR=8.10
#   하한=79.05, 상한=111.45
#   이상치 후보: 10 건 / 그중 불량: 2

### 실전 예고 — 이 규약이 처음 무너지는 곳 (필수, 5분)

In [ ]:
wmg = pd.read_csv('data/wmg/wmg_cells_54.csv')
print("모양:", wmg.shape)
print("결측 총 개수:", int(wmg.isna().sum().sum()))
print("dropna() 후:", wmg.dropna().shape)

# 교재 실행 결과 ------------------------------------------------------
#   모양: (54, 58)
#   결측 총 개수: 14
#   dropna() 후: (48, 58)

In [ ]:
print(wmg['grav_dis_10c_mahg'].describe().round(4))

# 교재 실행 결과 ------------------------------------------------------
#   count    54.0000
#   mean     42.0449
#   std      34.6477
#   min       0.0009
#   25%      12.2866
#   50%      27.5047
#   75%      78.8856
#   max      98.2473
#   Name: grav_dis_10c_mahg, dtype: float64